# Build the Older-Adult Depression and Anxiety Cohort

This notebook converts raw MIMIC-IV hospital and discharge-note tables into a
cleaned, labelled cohort for depression and anxiety modelling in adults aged
60 years or older.

It performs the following steps:

1. Locate and validate the required MIMIC-IV files.
2. Calculate age at admission and retain older-adult admissions.
3. Derive depression and anxiety labels from ICD-9 and ICD-10 codes.
4. Stream discharge summaries in memory-safe chunks.
5. Clean the text and optionally retain selected clinical sections.
6. Keep one discharge summary per admission.
7. Create patient-level train, validation and test splits.
8. Save the cohort and aggregate audit reports.

## 1. Import libraries


In [58]:
import gc
import json
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from IPython.display import display


## 2. Configure paths and cohort settings

`TOKENIZER_NAME` is optional. When it is `None`, token length is estimated as
1.4 tokens per word. When a tokenizer is provided, it is used only to measure
the selected text during cohort construction; token IDs are not saved.


In [59]:
# ============================================================
# CONFIGURE MACHINE-INDEPENDENT PATHS
# ============================================================

from pathlib import Path
import pandas as pd

from IPython.display import display


def find_project_root(starting_path):
    starting_path = Path(starting_path).expanduser().resolve()
    for location in [starting_path, *starting_path.parents]:
        if (location / ".git").exists() or (location / "dataset").exists():
            return location
    raise FileNotFoundError(
        "Could not locate the project root. Open Jupyter from inside "
        "the project repository."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_DIR = PROJECT_ROOT / "dataset"
HOSP_DIR = DATASET_DIR / "hosp"
NOTE_DIR = DATASET_DIR / "note"
OUTPUT_DIR = DATASET_DIR / "cohort_output"


# Cohort settings
MIN_AGE = 60
CHUNK_SIZE = 10_000
SEED = 42
SECTION_PROFILE = "clinical"  # "clinical", "minimal", or "none"
TOKENIZER_NAME = None         # Optional reference tokenizer
SAMPLE_SIZE = None           # Use None for the complete cohort
EXCLUDE_BIPOLAR = False
KEEP_TEMPORARY_FILE = False


if SECTION_PROFILE not in {"clinical", "minimal", "none"}:
    raise ValueError(
        "SECTION_PROFILE must be clinical, minimal, or none."
    )


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_NOTES_PATH = OUTPUT_DIR / "_notes_tmp.parquet"


configuration = pd.DataFrame(
    {
        "setting": [
            "Project root",
            "Hospital directory",
            "Hospital directory exists",
            "Note directory",
            "Note directory exists",
            "Output directory",
            "Minimum age",
            "Chunk size",
            "Section profile",
            "Reference tokenizer",
            "Sample size",
            "Exclude bipolar admissions",
            "Random seed",
        ],
        "value": [
            PROJECT_ROOT,
            HOSP_DIR,
            HOSP_DIR.exists(),
            NOTE_DIR,
            NOTE_DIR.exists(),
            OUTPUT_DIR,
            MIN_AGE,
            CHUNK_SIZE,
            SECTION_PROFILE,
            TOKENIZER_NAME,
            SAMPLE_SIZE,
            EXCLUDE_BIPOLAR,
            SEED,
        ],
    }
)


display(configuration)


,setting,value
0,Project root,/Users/purnaperamune/Documents/GitHub/P000464S...
1,Hospital directory,/Users/purnaperamune/Documents/GitHub/P000464S...
2,Hospital directory exists,True
3,Note directory,/Users/purnaperamune/Documents/GitHub/P000464S...
4,Note directory exists,True
5,Output directory,/Users/purnaperamune/Documents/GitHub/P000464S...
6,Minimum age,60
7,Chunk size,10000
8,Section profile,clinical
9,Reference tokenizer,None


## 3. Define labels, leakage patterns and output columns

The labels are derived only from the specified ICD code families. Medication
mentions and explicit condition terms are measured for audit purposes; they do
not determine the labels.


In [60]:
# --------------------------------------------------------------------------
# 1. LABEL DEFINITIONS  -- core codes only, no extensions, no inference
# --------------------------------------------------------------------------

DEPRESSION = {
    "icd9":  (r"^2962", r"^2963", r"^3004", r"^311"),
    "icd10": (r"^F32", r"^F33", r"^F341"),
}

ANXIETY = {
    "icd9":  (r"^3000", r"^3002"),
    "icd10": (r"^F40", r"^F41"),
}

# Not a label. Only used if --exclude-bipolar is passed.
BIPOLAR = (r"^2960", r"^2961", r"^2964", r"^2965", r"^2966", r"^2967", r"^F31")

# Reported covariate only -- never contributes to a label.
# Restricted to agents prescribed almost exclusively for depression or anxiety.
# Deliberately excluded, with reason:
#   amitriptyline / nortriptyline / doxepin  -> neuropathic pain, migraine, sleep
#   mirtazapine                              -> appetite stimulation, sleep
#   bupropion                                -> smoking cessation
#   duloxetine                               -> neuropathic pain, fibromyalgia
#   trazodone                                -> insomnia (dominant inpatient use)
#   all benzodiazepines                      -> sedation, withdrawal, procedures
ANTIDEPRESSANT_PATTERN = re.compile(
    r"\b(?:sertraline|fluoxetine|paroxetine|citalopram|escitalopram|fluvoxamine|"
    r"venlafaxine|desvenlafaxine|vortioxetine|vilazodone|buspirone|"
    r"phenelzine|tranylcypromine|"
    r"zoloft|prozac|paxil|celexa|lexapro|luvox|effexor|pristiq|trintellix|"
    r"viibryd|buspar|nardil|parnate)\b",
    re.IGNORECASE,
)


DEP_LEAK_PATTERN = re.compile(
    r"\b(?:depress\w*|dysthymi\w*|MDD|major depressive)\b",
    re.IGNORECASE,
)

ANX_LEAK_PATTERN = re.compile(
    r"\b(?:anxi\w*|panic|phobi\w*|agoraphob\w*|GAD|generali[sz]ed anxiety)\b",
    re.IGNORECASE,
)

# Union, used for masking at training time. Both labels are masked together so
# a multi-label model cannot read either condition off the text.
LEAK_PATTERN = re.compile(
    f"(?:{DEP_LEAK_PATTERN.pattern})|(?:{ANX_LEAK_PATTERN.pattern})",
    re.IGNORECASE,
)

NOTE_COLS = ["note_id", "subject_id", "hadm_id", "note_type", "charttime", "text"]

DEID_PATTERN = re.compile(r"_{2,}|\[\*\*.*?\*\*\]")
WS_PATTERN = re.compile(r"[ \t]+")
NL_PATTERN = re.compile(r"\n{3,}")

FINAL_COLS = ["hadm_id", "subject_id", "age_at_admission", "gender",
              "depression", "anxiety", "n_tokens", "text"]


## 4. Define text cleaning and section extraction

The `clinical` profile keeps seven selected sections. The `minimal` profile
keeps four shorter sections. The `none` profile keeps the complete cleaned
note. If selected headings cannot be identified, the full note is retained as
a fallback.


In [61]:
# --------------------------------------------------------------------------
# 1b. SECTIONS
# --------------------------------------------------------------------------

# A MIMIC section header: start of line, title-ish words, colon. This is why
# clean_text must not collapse \s+ to a single space -- flattening the note to
# one line destroys every header anchor.
HEADER = re.compile(r"^[ \t]*([A-Z][A-Za-z][A-Za-z /,'&\-]{1,58}):", re.MULTILINE)

SECTION_PROFILES = {
    "clinical": [
        "past medical history",
        "chief complaint",
        "history of present illness",
        "social history",
        "medications on admission",
        "discharge medications",
        "brief hospital course",
    ],
    "minimal": [
        "past medical history",
        "history of present illness",
        "social history",
        "medications on admission",
    ],
}


def split_sections(text):
    """{normalised_header: body}. Text before the first header goes to
    '(preamble)'. Repeated headers are concatenated."""
    out = defaultdict(list)
    marks = list(HEADER.finditer(text))
    if not marks:
        return {"(preamble)": text}
    if marks[0].start() > 0:
        out["(preamble)"].append(text[:marks[0].start()])
    for i, m in enumerate(marks):
        end = marks[i + 1].start() if i + 1 < len(marks) else len(text)
        key = " ".join(m.group(1).split()).lower()
        out[key].append(text[m.end():end])
    return {k: "\n".join(v) for k, v in out.items()}


def extract_sections(text, profile="clinical"):
    """Whitelisted sections only, in whitelist order. Returns the original
    text if nothing matched, so an unparseable note degrades rather than
    silently emptying."""
    if not profile or profile == "none":
        return text
    if not isinstance(text, str) or not text:
        return ""
    secs = split_sections(text)
    order = SECTION_PROFILES[profile]
    parts = [f"{n}: {secs[n].strip()}" for n in order if secs.get(n, "").strip()]
    return "\n\n".join(parts) if parts else text


def compile_set(patterns):
    return re.compile("|".join(patterns))


def match_codes(df, icd9_pats, icd10_pats):
    """Apply the ICD-9 patterns to ICD-9 rows and the ICD-10 patterns to
    ICD-10 rows, using the icd_version column rather than the admission date.
    Each branch is evaluated once, over its own subset only."""
    out = np.zeros(len(df), dtype=bool)
    is9 = (df["icd_version"] == 9).to_numpy()
    if is9.any():
        out[is9] = (df.loc[is9, "code_clean"]
                      .str.match(compile_set(icd9_pats), na=False).to_numpy())
    if (~is9).any():
        out[~is9] = (df.loc[~is9, "code_clean"]
                       .str.match(compile_set(icd10_pats), na=False).to_numpy())
    return out


def clean_text(t):
    """Light touch. Section headers are preserved -- section filtering,
    Longformer and the attention analysis all rely on document structure."""
    if not isinstance(t, str):
        return ""
    t = DEID_PATTERN.sub("[DEID]", t)    
    t = WS_PATTERN.sub(" ", t)
    t = NL_PATTERN.sub("\n\n", t)
    return t.strip()


def locate(directory, name):
    for ext in (".csv.gz", ".csv"):
        p = Path(directory) / f"{name}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"{name}.csv[.gz] not found in {directory}")


## 5. Define admission and ICD-label loading functions


In [62]:
# --------------------------------------------------------------------------
# 2. SMALL TABLES
# --------------------------------------------------------------------------

def load_admissions(hosp_dir, min_age):
    print("[1/6] patients + admissions", flush=True)
    patients = pd.read_csv(
        locate(hosp_dir, "patients"),
        usecols=["subject_id", "gender", "anchor_age", "anchor_year"],
    )
    admissions = pd.read_csv(
        locate(hosp_dir, "admissions"),
        usecols=["subject_id", "hadm_id", "admittime"],
        parse_dates=["admittime"],
    )
    adm = admissions.merge(patients, on="subject_id", how="inner")
    # Per-admission age, not patient-level anchor_age.
    adm["age_at_admission"] = (
        adm["anchor_age"] + (adm["admittime"].dt.year - adm["anchor_year"])
    )
    n0 = len(adm)
    adm = adm[adm["age_at_admission"] >= min_age].copy()
    # anchor_age is masked at 91 for anyone over 89 -- flag for the report only
    adm["age_capped"] = adm["age_at_admission"] >= 91
    print(f"      {n0:,} admissions -> {len(adm):,} at age >= {min_age}", flush=True)
    return adm[["subject_id", "hadm_id", "gender", "age_at_admission", "age_capped"]]


def load_labels(hosp_dir, keep_hadm, exclude_bipolar):
    print("[2/6] diagnoses -> binary labels", flush=True)
    dx = pd.read_csv(
        locate(hosp_dir, "diagnoses_icd"),
        usecols=["hadm_id", "icd_code", "icd_version"],
        dtype={"icd_code": str},
    )
    dx = dx[dx["hadm_id"].isin(keep_hadm)].copy()
    dx["code_clean"] = (
        dx["icd_code"].str.upper().str.replace(".", "", regex=False).str.strip()
    )

    dx["depression"] = match_codes(dx, DEPRESSION["icd9"], DEPRESSION["icd10"])
    dx["anxiety"] = match_codes(dx, ANXIETY["icd9"], ANXIETY["icd10"])

    cols = ["depression", "anxiety"]
    matched = {
        lab: dx.loc[dx[lab], "code_clean"].value_counts().head(40).to_dict()
        for lab in cols
    }

    if exclude_bipolar:
        dx["bipolar"] = dx["code_clean"].str.match(compile_set(BIPOLAR), na=False)
        cols = cols + ["bipolar"]

    labels = dx.groupby("hadm_id")[cols].any()
    del dx
    gc.collect()
    return labels.reset_index(), matched


## 6. Define memory-safe discharge-note streaming

The discharge table is read in chunks. Clinical text is written immediately to
a temporary Parquet file while smaller numeric metadata remains in memory.


In [63]:
# --------------------------------------------------------------------------
# 3. STREAMING NOTE PASS
# --------------------------------------------------------------------------

def stream_notes(note_dir, keep_hadm, tmp_path, chunksize=10_000,
                 tokenizer_name=None, sample=None, sections="clinical"):
    """Read discharge.csv in chunks. Clean each chunk, cut it down to the
    whitelisted sections, write the text straight to a temporary parquet, and
    keep only numeric metadata in RAM."""
    print("[3/6] discharge summaries (streaming)", flush=True)
    print(f"      section profile: {sections}", flush=True)
    path = locate(note_dir, "discharge")
    keep = set(keep_hadm)

    tok = None
    if tokenizer_name:
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(tokenizer_name)
        print(f"      tokenizer: {tokenizer_name}", flush=True)

    writer = None
    meta_parts = []
    seen = kept = 0

    reader = pd.read_csv(path, usecols=NOTE_COLS, chunksize=chunksize,
                         dtype={"text": str})
    try:
        for i, chunk in enumerate(reader, 1):
            seen += len(chunk)
            sub = chunk[chunk["hadm_id"].isin(keep)].copy()
            del chunk

            if len(sub):
                full = sub["text"].map(clean_text)
                sub["text"] = full.map(lambda t: extract_sections(t, sections))
                # no whitelisted header found -- full note kept as fallback
                sub["section_fallback"] = (sub["text"] == full)
                sub["n_words_full"] = full.str.split().str.len()
                del full

                sub["n_words"] = sub["text"].str.split().str.len()
                sub["char_len"] = sub["text"].str.len()
                sub["on_antidepressant"] = sub["text"].str.contains(ANTIDEPRESSANT_PATTERN)
                # counted separately per label -- a depression-positive note
                # saying "denies anxiety" must not score as a depression leak
                sub["n_leak_depression"] = sub["text"].str.count(DEP_LEAK_PATTERN)
                sub["n_leak_anxiety"] = sub["text"].str.count(ANX_LEAK_PATTERN)

                if tok is not None:
                    enc = tok(sub["text"].tolist(), truncation=False,
                              add_special_tokens=True)
                    sub["n_tokens"] = [len(x) for x in enc["input_ids"]]
                    del enc
                else:
                    # ~1.4 wordpiece tokens per word for clinical text
                    sub["n_tokens"] = (sub["n_words"] * 1.4).round().astype(int)
                sub["n_tokens_full"] = (sub["n_words_full"] * 1.4).round().astype(int)

                table = pa.Table.from_pandas(
                    sub[["hadm_id", "note_id", "text"]], preserve_index=False
                )
                if writer is None:
                    writer = pq.ParquetWriter(tmp_path, table.schema,
                                              compression="snappy")
                writer.write_table(table)
                del table

                meta_parts.append(sub.drop(columns=["text"]))
                kept += len(sub)
                del sub
                gc.collect()

            if i % 5 == 0:
                print(f"      scanned {seen:,}, kept {kept:,} ...", flush=True)
            if sample and kept >= sample:
                print(f"      --sample {sample} reached, stopping early", flush=True)
                break
    finally:
        if writer is not None:
            writer.close()

    meta = pd.concat(meta_parts, ignore_index=True)
    del meta_parts
    gc.collect()
    print(f"      scanned {seen:,} total, kept {kept:,}", flush=True)
    print(f"      median tokens {meta['n_tokens_full'].median():,.0f} -> "
          f"{meta['n_tokens'].median():,.0f}", flush=True)
    print(f"      no whitelisted section (kept full note): "
          f"{100 * meta['section_fallback'].mean():.1f}%", flush=True)
    return meta


## 7. Define patient-level splitting

Patients, rather than admissions, are assigned to train, validation and test
sets. This prevents different admissions belonging to the same patient from
appearing in multiple splits.


In [64]:
# --------------------------------------------------------------------------
# 4. SPLIT  -- returned as its own table, not merged into the cohort
# --------------------------------------------------------------------------

def patient_level_split(df, seed=42, val=0.15, test=0.15):
    """Split on subject_id -- a patient with several admissions must not
    straddle train and test. Discharge summaries restate a patient's history
    almost verbatim between visits, so an admission-level split lets a model
    score by recognising the patient rather than the condition.

    Stratified on the patient's label combination (neither / anxiety only /
    depression only / both), because PR-AUC is prevalence-sensitive and a
    drift in test prevalence would make a PR-AUC drop unreadable.

    Returns a (subject_id, split) frame. One row per patient.
    """
    rng = np.random.default_rng(seed)
    pat = df.groupby("subject_id")[["depression", "anxiety"]].any()
    pat["stratum"] = pat["depression"].astype(int) * 2 + pat["anxiety"].astype(int)

    assign = {}
    for _, grp in pat.groupby("stratum"):
        subs = grp.index.to_numpy().copy()
        rng.shuffle(subs)
        n_test, n_val = int(len(subs) * test), int(len(subs) * val)
        assign.update({s: "test" for s in subs[:n_test]})
        assign.update({s: "val" for s in subs[n_test:n_test + n_val]})
        assign.update({s: "train" for s in subs[n_test + n_val:]})

    return pd.DataFrame(
        {"subject_id": list(assign.keys()), "split": list(assign.values())}
    ).sort_values("subject_id", ignore_index=True)


## 8. Define final assembly and reporting functions


In [65]:
# --------------------------------------------------------------------------
# 5. FINAL ASSEMBLY  -- stream text back in, attach labels, write one file
# --------------------------------------------------------------------------

def assemble(tmp_path, meta, out_path, batch_size=5_000):
    print("[5/6] writing cohort.parquet (text + labels)", flush=True)
    key = meta.set_index("note_id")
    pf = pq.ParquetFile(tmp_path)
    writer = None
    written = 0

    try:
        for batch in pf.iter_batches(batch_size=batch_size,
                                     columns=["note_id", "text"]):
            part = batch.to_pandas()
            # inner join drops duplicate notes not kept for an admission
            part = part.join(key, on="note_id", how="inner")
            if not len(part):
                continue
            part = part[FINAL_COLS]
            table = pa.Table.from_pandas(part, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(out_path, table.schema,
                                          compression="snappy")
            writer.write_table(table)
            written += len(part)
            del part, table, batch
            gc.collect()
    finally:
        if writer is not None:
            writer.close()
        pf.close()          # Windows will not unlink a file with an open handle

    print(f"      wrote {written:,} rows", flush=True)
    return written


In [66]:
# --------------------------------------------------------------------------
# 6. REPORT
# --------------------------------------------------------------------------

def build_report(df, splits, matched_codes, n_bipolar_dropped, sections):
    d = df.merge(splits, on="subject_id", how="left")
    rep = {
        "n_admissions": int(len(d)),
        "n_patients": int(d["subject_id"].nunique()),
        "n_bipolar_admissions_dropped": int(n_bipolar_dropped),
        "age": {
            "median": float(d["age_at_admission"].median()),
            "iqr": [float(d["age_at_admission"].quantile(.25)),
                    float(d["age_at_admission"].quantile(.75))],
            "pct_age_capped_91": round(100 * d["age_capped"].mean(), 2),
        },
        "sex_pct": d["gender"].value_counts(normalize=True).mul(100).round(2).to_dict(),
        "sections": {
            "profile": sections,
            "kept": SECTION_PROFILES.get(sections, []),
            "median_tokens_before": int(d["n_tokens_full"].median()),
            "median_tokens_after": int(d["n_tokens"].median()),
            "pct_reduction": round(
                100 * (1 - d["n_tokens"].median() / max(d["n_tokens_full"].median(), 1)), 1),
            "pct_no_section_found": round(100 * d["section_fallback"].mean(), 2),
        },
        "note_length_tokens": {
            str(q): int(d["n_tokens"].quantile(q / 100))
            for q in (5, 25, 50, 75, 90, 95, 99)
        },
        "truncation": {
            "pct_over_512": round(100 * (d["n_tokens"] > 512).mean(), 2),
            "pct_over_1024": round(100 * (d["n_tokens"] > 1024).mean(), 2),
            "pct_over_4096": round(100 * (d["n_tokens"] > 4096).mean(), 2),
        },
        "split_sizes_admissions": d["split"].value_counts().to_dict(),
        "split_sizes_patients": splits["split"].value_counts().to_dict(),
        "matched_icd_codes_top40": matched_codes,
    }
    leak_col = {"depression": "n_leak_depression", "anxiety": "n_leak_anxiety"}
    for label in ("depression", "anxiety"):
        pos = d[label].astype(bool)
        own = d[leak_col[label]] > 0
        rep[label] = {
            "n_positive": int(pos.sum()),
            "prevalence_pct": round(100 * pos.mean(), 2),
            "prevalence_by_split_pct": (
                d.groupby("split")[label].mean().mul(100).round(2).to_dict()
            ),
            # own-condition terms only, measured on the filtered text
            "pct_positives_with_explicit_mention": round(100 * own[pos].mean(), 2),
            "pct_negatives_with_explicit_mention": round(100 * own[~pos].mean(), 2),
        }
    # under-diagnosis signal, conditioned on being negative for BOTH labels so
    # a depression patient on sertraline does not inflate the anxiety figure
    neither = ~d["depression"].astype(bool) & ~d["anxiety"].astype(bool)
    rep["undiagnosed"] = {
        "n_negative_for_both": int(neither.sum()),
        "pct_on_antidepressant": round(
            100 * d.loc[neither, "on_antidepressant"].mean(), 2),
        "pct_with_any_explicit_mention": round(
            100 * ((d.loc[neither, "n_leak_depression"]
                    + d.loc[neither, "n_leak_anxiety"]) > 0).mean(), 2),
    }
    rep["comorbid_both_pct"] = round(
        100 * (d["depression"].astype(bool) & d["anxiety"].astype(bool)).mean(), 2
    )
    return rep


## 9. Define the optional training-data loader

This helper can attach the patient split, mask explicit condition terms and
optionally truncate text. For the benchmarking pipeline, leaving
`truncate=None` is recommended so each model notebook can apply its own
tokenizer and chunking strategy.


In [67]:
# --------------------------------------------------------------------------
# 7. TRAIN-TIME LOADER
# --------------------------------------------------------------------------

def _truncate(text, tok, mode, max_tokens, head_frac=0.25):
    if mode in (None, "none", "full"):
        return text
    if tok is not None:
        ids = tok.encode(text, add_special_tokens=False)
        if len(ids) <= max_tokens:
            return text
        if mode == "head":
            keep = ids[:max_tokens]
        elif mode == "tail":
            keep = ids[-max_tokens:]
        else:  # head_tail
            h = int(max_tokens * head_frac)
            keep = ids[:h] + ids[-(max_tokens - h):]
        return tok.decode(keep)
    # word-level fallback: ~1.4 tokens per word
    words = text.split()
    budget = int(max_tokens / 1.4)
    if len(words) <= budget:
        return text
    if mode == "head":
        return " ".join(words[:budget])
    if mode == "tail":
        return " ".join(words[-budget:])
    h = int(budget * head_frac)
    return " ".join(words[:h] + words[-(budget - h):])


def load_splits(out_dir):
    """The (subject_id, split) table. Small -- hand this to a collaborator so
    both of you score on the same patients."""
    return pd.read_parquet(Path(out_dir) / "splits.parquet")


def load_for_training(out_dir, split=None, mask_leakage=False,
                      truncate=None, max_tokens=512, tokenizer_name=None,
                      columns=None, allow_test=False):
    """Read the cohort and attach the split. Masking and truncation are applied
    here rather than stored, so there is only ever one copy of the notes on
    disk and the raw/masked and head/tail/full ablations stay available.

    test is withheld unless allow_test=True. Tune on val; run test once.
    """
    if split == "test" and not allow_test:
        raise RuntimeError(
            "test split withheld. Every tuning and ablation decision should be "
            "made on val -- scoring 30 configurations on test and reporting the "
            "best turns it into a selection set. Pass allow_test=True for the "
            "single final evaluation."
        )

    out = Path(out_dir)
    splits = load_splits(out)

    filters = None
    if split:
        subs = splits.loc[splits["split"] == split, "subject_id"]
        # pushed down, so the text of the other splits is never read
        filters = [("subject_id", "in", set(subs.tolist()))]

    df = pd.read_parquet(out / "cohort.parquet", filters=filters)
    df = df.merge(splits, on="subject_id", how="left")

    if mask_leakage:
        df["text"] = df["text"].str.replace(LEAK_PATTERN, "[MASK]", regex=True)

    if truncate:
        tok = None
        if tokenizer_name:
            from transformers import AutoTokenizer
            tok = AutoTokenizer.from_pretrained(tokenizer_name)
        df["text"] = df["text"].map(
            lambda t: _truncate(t, tok, truncate, max_tokens)
        )

    return df[columns] if columns else df


## 10. Validate input files


In [68]:
# ============================================================
# VALIDATE REQUIRED INPUT FILES
# ============================================================

required_files = {
    "patients": locate(HOSP_DIR, "patients"),
    "admissions": locate(HOSP_DIR, "admissions"),
    "diagnoses_icd": locate(HOSP_DIR, "diagnoses_icd"),
    "discharge": locate(NOTE_DIR, "discharge"),
}


input_file_summary = pd.DataFrame(
    [
        {
            "table": table_name,
            "path": file_path,
            "exists": file_path.exists(),
            "size_gb": round(file_path.stat().st_size / (1024 ** 3), 3),
        }
        for table_name, file_path in required_files.items()
    ]
)


display(input_file_summary)


if not input_file_summary["exists"].all():
    raise FileNotFoundError("One or more required MIMIC-IV files are missing.")


print("All required input files were found.")

,table,path,exists,size_gb
0,patients,/Users/purnaperamune/Documents/GitHub/P000464S...,True,0.011
1,admissions,/Users/purnaperamune/Documents/GitHub/P000464S...,True,0.019
2,diagnoses_icd,/Users/purnaperamune/Documents/GitHub/P000464S...,True,0.031
3,discharge,/Users/purnaperamune/Documents/GitHub/P000464S...,True,1.061


All required input files were found.


## 11. Build the older-adult admission cohort


In [69]:
admissions_older = load_admissions(
    HOSP_DIR,
    MIN_AGE,
)


display(admissions_older.head())


print(
    "Older-adult admissions:",
    f"{len(admissions_older):,}",
)


[1/6] patients + admissions
      546,028 admissions -> 287,702 at age >= 60


,subject_id,hadm_id,gender,age_at_admission,age_capped
5,10000084,23052089,M,72,False
6,10000084,29888819,M,72,False
10,10000161,22148160,M,60,False
14,10000635,20642640,F,81,False
15,10000635,26134563,F,74,False


Older-adult admissions: 287,702


## 12. Derive depression and anxiety labels


In [70]:
labels, matched_codes = load_labels(
    HOSP_DIR,
    set(admissions_older["hadm_id"]),
    EXCLUDE_BIPOLAR,
)


display(labels.head())


print(
    "Admissions with at least one diagnoses_icd record:",
    f"{len(labels):,}",
)


[2/6] diagnoses -> binary labels


,hadm_id,depression,anxiety
0,20000019,False,False
1,20000024,False,False
2,20000034,False,False
3,20000041,False,False
4,20000057,False,False


Admissions with at least one diagnoses_icd record: 287,496


## 13. Stream and prepare discharge summaries

This is the longest-running step. With `SECTION_PROFILE="clinical"` or
`"minimal"`, the saved `text` will contain only matched sections. With
`SECTION_PROFILE="none"`, the complete cleaned note is retained.


In [71]:
note_metadata = stream_notes(
    NOTE_DIR,
    set(admissions_older["hadm_id"]),
    TEMP_NOTES_PATH,
    chunksize=CHUNK_SIZE,
    tokenizer_name=TOKENIZER_NAME,
    sample=SAMPLE_SIZE,
    sections=SECTION_PROFILE,
)


display(note_metadata.head())


[3/6] discharge summaries (streaming)
      section profile: clinical
      scanned 50,000, kept 28,694 ...
      scanned 100,000, kept 57,523 ...
      scanned 150,000, kept 86,217 ...
      scanned 200,000, kept 115,121 ...
      scanned 250,000, kept 143,981 ...
      scanned 300,000, kept 172,751 ...
      scanned 331,793 total, kept 191,084
      median tokens 2,212 -> 798
      no whitelisted section (kept full note): 0.0%


,note_id,subject_id,hadm_id,note_type,charttime,section_fallback,n_words_full,n_words,char_len,on_antidepressant,n_leak_depression,n_leak_anxiety,n_tokens,n_tokens_full
0,10000084-DS-17,10000084,23052089,DS,2160-11-25 00:00:00,False,1396,127,848,False,0,0,178,1954
1,10000764-DS-11,10000764,27897940,DS,2132-10-19 00:00:00,False,1462,848,5572,False,0,0,1187,2047
2,10000980-DS-20,10000980,29654838,DS,2188-01-05 00:00:00,False,1984,913,5646,False,0,0,1278,2778
3,10000980-DS-21,10000980,26913865,DS,2189-07-03 00:00:00,False,1755,663,4257,False,0,0,928,2457
4,10000980-DS-22,10000980,24947999,DS,2190-11-08 00:00:00,False,2180,1127,6947,False,1,0,1578,3052


## 14. Keep one note per admission and join the cohort

When multiple discharge summaries exist for an admission, the longest selected
text is retained. Missing depression and anxiety labels are assigned zero,
meaning no matching code was found in `diagnoses_icd`.


In [72]:
note_metadata = (
    note_metadata
    .sort_values("char_len", ascending=False)
    .drop_duplicates("hadm_id", keep="first")
)


cohort_metadata = (
    admissions_older
    .merge(labels, on="hadm_id", how="left")
    .merge(
        note_metadata.drop(columns=["subject_id"]),
        on="hadm_id",
        how="inner",
    )
)


number_bipolar_dropped = 0


if EXCLUDE_BIPOLAR:
    cohort_metadata["bipolar"] = (
        cohort_metadata["bipolar"]
        .fillna(False)
        .astype(bool)
    )
    number_bipolar_dropped = int(cohort_metadata["bipolar"].sum())
    cohort_metadata = (
        cohort_metadata.loc[~cohort_metadata["bipolar"]]
        .drop(columns=["bipolar"])
        .copy()
    )


for label_column in ("depression", "anxiety"):
    cohort_metadata[label_column] = (
        cohort_metadata[label_column]
        .fillna(False)
        .astype(int)
    )


display(
    cohort_metadata[
        [
            "hadm_id",
            "subject_id",
            "age_at_admission",
            "gender",
            "depression",
            "anxiety",
            "n_tokens",
        ]
    ].head()
)


print("Cohort admissions:", f"{len(cohort_metadata):,}")
print("Cohort patients:", f"{cohort_metadata['subject_id'].nunique():,}")


,hadm_id,subject_id,age_at_admission,gender,depression,anxiety,n_tokens
0,23052089,10000084,72,M,0,0,178
1,27897940,10000764,86,M,0,0,1187
2,20897796,10000980,80,F,0,0,619
3,24947999,10000980,77,F,0,0,1578
4,25242409,10000980,78,F,0,0,545


Cohort admissions: 191,084
Cohort patients: 80,087


## 15. Create and save patient-level splits


In [73]:
splits = patient_level_split(
    cohort_metadata,
    seed=SEED,
)


splits.to_parquet(
    OUTPUT_DIR / "splits.parquet",
    index=False,
)


split_summary = (
    splits["split"]
    .value_counts()
    .rename_axis("split")
    .reset_index(name="patients")
)


display(split_summary)


,split,patients
0,train,56065
1,test,12011
2,val,12011


## 16. Write the final cohort and audit reports


In [74]:
number_written = assemble(
    TEMP_NOTES_PATH,
    cohort_metadata,
    OUTPUT_DIR / "cohort.parquet",
)


cohort_report = build_report(
    cohort_metadata,
    splits,
    matched_codes,
    number_bipolar_dropped,
    SECTION_PROFILE,
)


(OUTPUT_DIR / "cohort_stats.json").write_text(
    json.dumps(
        cohort_report,
        indent=2,
        default=str,
    )
)


histogram_counts, histogram_edges = np.histogram(
    cohort_metadata["n_tokens"],
    bins=50,
)


pd.DataFrame(
    {
        "bin_left": histogram_edges[:-1],
        "bin_right": histogram_edges[1:],
        "count": histogram_counts,
    }
).to_csv(
    OUTPUT_DIR / "note_length_hist.csv",
    index=False,
)


if not KEEP_TEMPORARY_FILE:
    try:
        TEMP_NOTES_PATH.unlink(missing_ok=True)
    except OSError as error:
        print(
            "Temporary file could not be removed:",
            error,
        )


print("Cohort rows written:", f"{number_written:,}")
print("Output directory:", OUTPUT_DIR)


[5/6] writing cohort.parquet (text + labels)
      wrote 191,084 rows
Cohort rows written: 191,084
Output directory: /Users/purnaperamune/Documents/GitHub/P000464SE-Modern-Transformer-Models-for-Depression-Anxiety-Detection/dataset/cohort_output


## 17. Inspect and validate the outputs

The text remains readable text. `n_tokens` is only a length measurement. The
final model notebook must still apply the model's tokenizer and construct token
IDs, attention masks, chunks and padded batches.


In [75]:
output_files = [
    OUTPUT_DIR / "cohort.parquet",
    OUTPUT_DIR / "splits.parquet",
    OUTPUT_DIR / "cohort_stats.json",
    OUTPUT_DIR / "note_length_hist.csv",
]


output_summary = pd.DataFrame(
    {
        "file": [path.name for path in output_files],
        "exists": [path.exists() for path in output_files],
        "size_mb": [
            round(path.stat().st_size / (1024 ** 2), 3)
            if path.exists()
            else np.nan
            for path in output_files
        ],
    }
)


display(output_summary)


cohort_preview = pd.read_parquet(
    OUTPUT_DIR / "cohort.parquet"
).head(5)


display(cohort_preview)


if cohort_preview["text"].isna().any():
    raise ValueError("The cohort preview contains missing text.")


if not set(cohort_preview["depression"].unique()).issubset({0, 1}):
    raise ValueError("Unexpected depression label detected.")


if not set(cohort_preview["anxiety"].unique()).issubset({0, 1}):
    raise ValueError("Unexpected anxiety label detected.")


print("Initial output validation passed.")


,file,exists,size_mb
0,cohort.parquet,True,359.859
1,splits.parquet,True,0.522
2,cohort_stats.json,True,0.003
3,note_length_hist.csv,True,0.001


,hadm_id,subject_id,age_at_admission,gender,depression,anxiety,n_tokens,text
0,23052089,10000084,72,M,0,0,178,past medical history: [DEID] disease\n[DEID] B...
1,27897940,10000764,86,M,0,0,1187,past medical history: MI after AAA repair when...
2,29654838,10000980,75,F,0,0,1278,past medical history: 1. CAD RISK FACTORS: +Di...
3,26913865,10000980,76,F,0,0,928,past medical history: - hypertension\n- diabet...
4,24947999,10000980,77,F,0,0,1578,past medical history: - hypertension \n- diabe...


Initial output validation passed.


## 18. Example: load training text without truncating it

This example attaches the training split and masks explicit depression and
anxiety terms in memory. It leaves truncation and tokenisation to the model
notebook. The stored `cohort.parquet` is not modified.


In [76]:
training_data = load_for_training(
    OUTPUT_DIR,
    split="train",
    mask_leakage=True,
    truncate=None,
)


print("Training admissions:", f"{len(training_data):,}")


display(
    training_data[
        [
            "hadm_id",
            "subject_id",
            "age_at_admission",
            "depression",
            "anxiety",
            "n_tokens",
            "split",
        ]
    ].head()
)


Training admissions: 133,793


,hadm_id,subject_id,age_at_admission,depression,anxiety,n_tokens,split
0,23052089,10000084,72,0,0,178,train
1,27897940,10000764,86,0,0,1187,train
2,22672901,10001667,86,0,0,1344,train
3,25679292,10001877,89,0,0,1113,train
4,21320596,10001877,90,0,0,1114,train
